# 04. W6 Model Training and Feature Analysis

**Objective**: Execute end-to-end model training pipeline, feature importance analysis, and VG Node 1 auxiliary feature impact documentation.

**Week 6 Tasks**:
- [x] Initial Model Training Run (RF, XGBoost, LightGBM)
- [x] Feature Importance Analysis
- [x] Cross-Validation Results
- [x] Model Persistence
- [x] VG Node 1: Auxiliary Feature Impact Analysis

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features import DataProcessor
from src.models import (
    RandomForestForecaster, 
    XGBoostForecaster, 
    LightGBMForecaster,
    compare_models,
    create_train_test_split
)
from src.utils import logger, load_data, DATA_PATHS

# Visualization settings
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Imports successful")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Load pre-computed features or generate them
try:
    # Try to load existing features first
    df = load_data('processed', 'se3_features_v1.parquet')
    print(f"✓ Loaded existing features: {df.shape}")
except FileNotFoundError:
    print("Generating features...")
    processor = DataProcessor(target_area='SE3')
    df = processor.prepare_features()
    processor.save_processed_data(df, 'se3_features_v1.parquet')
    print(f"✓ Generated and saved features: {df.shape}")

print(f"\nData range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Features: {len(df.columns)} columns")
print(f"\nFeature list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

## 2. Train/Test Split (Time-Series Aware)

In [ ]:
# 80/20 chronological split
X_train, y_train, X_test, y_test = create_train_test_split(
    df, 
    target_col='value', 
    train_ratio=0.8,
    drop_cols=[]  # timestamp will be auto-dropped
)

print("\n" + "="*50)
print("DATASET SPLIT SUMMARY")
print("="*50)
print(f"Training set:   {len(X_train):,} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test set:       {len(X_test):,} samples ({len(X_test)/len(df)*100:.1f}%)")
print(f"\nFeatures:       {X_train.shape[1]}")
print(f"\nTarget stats:")
print(f"  Train mean:   {y_train.mean():.2f} SEK/MWh")
print(f"  Test mean:    {y_test.mean():.2f} SEK/MWh")
print(f"  Train std:    {y_train.std():.2f} SEK/MWh")
print(f"  Test std:     {y_test.std():.2f} SEK/MWh")

## 3. Model Training

In [ ]:
# Initialize all three models
models = {
    'Random Forest': RandomForestForecaster(
        n_estimators=100, 
        max_depth=10,
        random_state=42
    ),
    'XGBoost': XGBoostForecaster(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ),
    'LightGBM': LightGBMForecaster(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        random_state=42
    )
}

# Train all models
print("\n" + "="*50)
print("TRAINING MODELS")
print("="*50)

for name, model in models.items():
    print(f"\n{'-'*50}")
    print(f"Training {name}...")
    model.fit(X_train, y_train, X_test, y_test)

## 4. Model Evaluation and Comparison

In [ ]:
# Generate predictions for each model
predictions = {}
results = []

for name, model in models.items():
    preds = model.predict(X_test)
    predictions[name] = preds
    
    metrics = model.evaluate(X_test, y_test, verbose=False)
    results.append({
        'Model': name,
        'MAE': metrics['mae'],
        'RMSE': metrics['rmse'],
        'MAPE (%)': metrics['mape']
    })

# Display comparison table
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("MODEL PERFORMANCE COMPARISON")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

# Identify best model
best_model_name = results_df.loc[results_df['MAE'].idxmin(), 'Model']
print(f"\n✓ Best model: {best_model_name} (lowest MAE)")

In [ ]:
# Visualization: Actual vs Predicted
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, preds) in enumerate(predictions.items()):
    ax = axes[idx]
    
    # Scatter plot
    ax.scatter(y_test.iloc[::10], preds[::10], alpha=0.3, s=1)
    
    # Perfect prediction line
    min_val = min(y_test.min(), preds.min())
    max_val = max(y_test.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect')
    
    ax.set_xlabel('Actual Price (SEK/MWh)')
    ax.set_ylabel('Predicted Price (SEK/MWh)')
    ax.set_title(f'{name}\nActual vs Predicted')
    ax.legend()
    
    # R² score
    from sklearn.metrics import r2_score
    r2 = r2_score(y_test, preds)
    ax.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat'))

plt.tight_layout()
plt.savefig(DATA_PATHS['reports'] / 'figures' / '04_actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
# Time series prediction visualization (sample of test set)
sample_size = min(168, len(y_test))  # 1 week or less
sample_idx = slice(0, sample_size)

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(y_test.iloc[sample_idx].values, label='Actual', color='black', linewidth=2)
colors = ['#2ecc71', '#3498db', '#e74c3c']

for (name, preds), color in zip(predictions.items(), colors):
    ax.plot(preds[sample_idx], label=name, alpha=0.7, linewidth=1.5, color=color)

ax.set_xlabel('Hours')
ax.set_ylabel('Price (SEK/MWh)')
ax.set_title('Model Predictions vs Actual (Test Set Sample - First Week)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_PATHS['reports'] / 'figures' / '04_time_series_predictions.png', dpi=150)
plt.show()

## 5. Feature Importance Analysis

In [ ]:
# Get feature importance from all models
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for idx, (name, model) in enumerate(models.items()):
    importance = model.feature_importance()
    
    if importance is not None:
        # Top 15 features
        top_features = importance.head(15)
        
        ax = axes[idx]
        top_features.plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title(f'{name}\nTop 15 Feature Importance')
        ax.set_xlabel('Importance')
        ax.invert_yaxis()

plt.tight_layout()
plt.savefig(DATA_PATHS['reports'] / 'figures' / '04_feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed XGBoost feature importance (for VG report)
xgb_model = models['XGBoost']
xgb_importance = xgb_model.feature_importance()

print("\n" + "="*60)
print("XGBOOST FEATURE IMPORTANCE RANKING (Top 20)")
print("="*60)
for rank, (feature, importance) in enumerate(xgb_importance.head(20).items(), 1):
    marker = ""
    if 'wind' in feature.lower():
        marker = " [WEATHER]"
    elif 'temp' in feature.lower():
        marker = " [WEATHER]"
    elif 'holiday' in feature.lower():
        marker = " [HOLIDAY]"
    elif 'sin' in feature or 'cos' in feature:
        marker = " [CYCLICAL]"
    print(f"{rank:2d}. {feature:30s} {importance:.4f}{marker}")
print("="*60)

# Summary by category
weather_features = [f for f in xgb_importance.index if 'wind' in f.lower() or 'temp' in f.lower()]
holiday_features = [f for f in xgb_importance.index if 'holiday' in f.lower()]
cyclical_features = [f for f in xgb_importance.index if 'sin' in f or 'cos' in f]
lag_features = [f for f in xgb_importance.index if 'lag' in f.lower()]

print("\nFEATURE CATEGORY SUMMARY:")
print(f"  Weather features (temp, wind):  {len(weather_features)} features")
print(f"    Top 3 weather: {', '.join(weather_features[:3])}")
print(f"  Holiday features:               {len(holiday_features)} features")
print(f"    {', '.join(holiday_features)}")
print(f"  Cyclical features (sin/cos):    {len(cyclical_features)} features")
print(f"  Lag features:                   {len(lag_features)} features")

## 6. Cross-Validation Results

In [ ]:
# 5-fold time-series cross-validation
print("\n" + "="*60)
print("5-FOLD TIME-SERIES CROSS-VALIDATION")
print("="*60)

# Combine train and test for CV
X_full = pd.concat([X_train, X_test], ignore_index=True)
y_full = pd.concat([y_train, y_test], ignore_index=True)

cv_results = {}

for name, model in models.items():
    print(f"\n{'-'*60}")
    cv_summary = model.cross_validate(X_full, y_full, n_splits=5)
    cv_results[name] = cv_summary

# Summary table
print("\n" + "="*60)
print("CROSS-VALIDATION SUMMARY (Mean ± Std)")
print("="*60)

cv_df = pd.DataFrame({
    name: {
        'MAE': f"{r['cv_mae_mean']:.4f} ± {r['cv_mae_std']:.4f}",
        'RMSE': f"{r['cv_rmse_mean']:.4f} ± {r['cv_rmse_std']:.4f}",
        'MAPE': f"{r['cv_mape_mean']:.2f}% ± {r['cv_mape_std']:.2f}%"
    }
    for name, r in cv_results.items()
}).T

print(cv_df.to_string())
print("="*60)

## 7. VG Node 1: Auxiliary Feature Impact Analysis

**Objective**: Quantify the impact of weather and holiday auxiliary features on model performance.

In [ ]:
# Define feature sets for ablation study
all_features = list(X_train.columns)

weather_features = [f for f in all_features if any(x in f.lower() for x in ['temp', 'wind'])]
holiday_features = [f for f in all_features if 'holiday' in f.lower()]
cyclical_features = [f for f in all_features if any(x in f for x in ['_sin', '_cos'])]

print("Feature Categories:")
print(f"  Weather features:   {len(weather_features)} ({', '.join(weather_features)})")
print(f"  Holiday features:   {len(holiday_features)} ({', '.join(holiday_features)})")
print(f"  Cyclical features:  {len(cyclical_features)}")
print(f"  Total features:     {len(all_features)}")

In [ ]:
# Ablation study: Train models with different feature sets
feature_sets = {
    'All Features': all_features,
    'No Weather': [f for f in all_features if f not in weather_features],
    'No Holidays': [f for f in all_features if f not in holiday_features],
    'No Weather + No Holidays': [f for f in all_features if f not in weather_features + holiday_features],
    'Core Only (No Aux)': [f for f in all_features if f not in weather_features + holiday_features + cyclical_features]
}

print("\n" + "="*70)
print("ABLATION STUDY: AUXILIARY FEATURE IMPACT")
print("="*70)

ablation_results = []

for set_name, features in feature_sets.items():
    print(f"\n{'-'*70}")
    print(f"Feature Set: {set_name}")
    print(f"Feature Count: {len(features)}")
    
    X_train_sub = X_train[features]
    X_test_sub = X_test[features]
    
    # Train XGBoost for comparison
    model = XGBoostForecaster(n_estimators=500, learning_rate=0.01, random_state=42)
    model.fit(X_train_sub, y_train)
    metrics = model.evaluate(X_test_sub, y_test, verbose=False)
    
    ablation_results.append({
        'Feature Set': set_name,
        'N Features': len(features),
        'MAE': metrics['mae'],
        'RMSE': metrics['rmse'],
        'MAPE (%)': metrics['mape']
    })
    
    print(f"  MAE:  {metrics['mae']:.4f}")
    print(f"  RMSE: {metrics['rmse']:.4f}")
    print(f"  MAPE: {metrics['mape']:.2f}%")

# Create comparison DataFrame
ablation_df = pd.DataFrame(ablation_results)

print("\n" + "="*70)
print("ABLATION STUDY SUMMARY (XGBoost)")
print("="*70)
print(ablation_df.to_string(index=False))
print("="*70)

In [ ]:
# Calculate impact of auxiliary features
baseline_mae = ablation_df[ablation_df['Feature Set'] == 'All Features']['MAE'].values[0]
no_weather_mae = ablation_df[ablation_df['Feature Set'] == 'No Weather']['MAE'].values[0]
no_holidays_mae = ablation_df[ablation_df['Feature Set'] == 'No Holidays']['MAE'].values[0]
no_aux_mae = ablation_df[ablation_df['Feature Set'] == 'No Weather + No Holidays']['MAE'].values[0]

weather_impact = no_weather_mae - baseline_mae
holiday_impact = no_holidays_mae - baseline_mae
combined_impact = no_aux_mae - baseline_mae

print("\n" + "="*70)
print("VG NODE 1: AUXILIARY FEATURE IMPACT ANALYSIS")
print("="*70)
print(f"\nBaseline (All Features):     MAE = {baseline_mae:.4f} SEK/MWh")
print(f"\nImpact of removing features:")
print(f"  Weather features:          +{weather_impact:.4f} MAE ({weather_impact/baseline_mae*100:+.2f}%)")
print(f"  Holiday features:          +{holiday_impact:.4f} MAE ({holiday_impact/baseline_mae*100:+.2f}%)")
print(f"  Combined (Weather+Holidays): +{combined_impact:.4f} MAE ({combined_impact/baseline_mae*100:+.2f}%)")
print(f"\nConclusion:")
print(f"  Auxiliary features improve prediction accuracy by {combined_impact/baseline_mae*100:.2f}%.")
print(f"  Weather features contribute {weather_impact/combined_impact*100:.1f}% of this improvement.")
print(f"  Holiday features contribute {holiday_impact/combined_impact*100:.1f}% of this improvement.")
print("="*70)

In [ ]:
# Visualization: Ablation study results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# MAE comparison
ax1 = axes[0]
colors = ['#2ecc71' if 'All' in x else '#3498db' if 'No' in x else '#95a5a6' 
          for x in ablation_df['Feature Set']]
bars = ax1.barh(ablation_df['Feature Set'], ablation_df['MAE'], color=colors)
ax1.set_xlabel('MAE (SEK/MWh)')
ax1.set_title('Model Performance by Feature Set\n(Lower is Better)')
ax1.invert_yaxis()

# Add value labels
for bar, val in zip(bars, ablation_df['MAE']):
    ax1.text(val + 0.1, bar.get_y() + bar.get_height()/2, 
             f'{val:.2f}', va='center')

# Feature importance by category
ax2 = axes[1]

# Sum importance by category
xgb_imp = xgb_model.feature_importance()
category_importance = {
    'Core\n(Price Lags/Rolling)': sum(xgb_imp.get(f, 0) for f in all_features 
                                     if any(x in f for x in ['lag', 'rolling', 'diff'])),
    'Temporal\n(Hour/Month/DOW)': sum(xgb_imp.get(f, 0) for f in all_features
                                     if f in ['hour', 'month', 'day_of_week', 'is_weekend'] or
                                     'peak' in f),
    'Cyclical\n(Sin/Cos)': sum(xgb_imp.get(f, 0) for f in cyclical_features),
    'Weather\n(Temp/Wind)': sum(xgb_imp.get(f, 0) for f in weather_features),
    'Holidays': sum(xgb_imp.get(f, 0) for f in holiday_features)
}

ax2.pie(category_importance.values(), labels=category_importance.keys(), 
        autopct='%1.1f%%', startangle=90)
ax2.set_title('XGBoost Feature Importance by Category')

plt.tight_layout()
plt.savefig(DATA_PATHS['reports'] / 'figures' / '04_vg_node1_auxiliary_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Persistence

In [ ]:
# Save all trained models
print("\n" + "="*60)
print("SAVING MODELS")
print("="*60)

saved_models = {}

for name, model in models.items():
    filename = f"w6_{name.lower().replace(' ', '_')}_model.pkl"
    path = model.save(filename)
    saved_models[name] = path
    print(f"✓ {name}: {path.name}")

print("\n" + "="*60)
print("Saved model files:")
for name, path in saved_models.items():
    print(f"  {path}")
print("="*60)

In [ ]:
# Test model loading
print("\nTesting model loading...")

from src.models import XGBoostForecaster

loaded_model = XGBoostForecaster.load(saved_models['XGBoost'])
test_preds = loaded_model.predict(X_test.head(5))

print(f"✓ Loaded model prediction test: {test_preds}")
print(f"✓ Model ready for deployment")

## 9. Summary and Next Steps

In [ ]:
# Final summary
print("\n" + "="*70)
print("W6 DEVELOPMENT SUMMARY")
print("="*70)

print("\n✓ COMPLETED TASKS:")
print(
     "  1. Initial Model Training Run\n",
     "     - Random Forest (baseline)\n",
     "     - XGBoost (best performing)\n",
     "     - LightGBM (fast training)\")\n",)


print("  2. Feature Importance Analysis")
top3 = xgb_importance.head(3)
for i, (feat, imp) in enumerate(top3.items(), 1):
    print(f"     {i}. {feat} ({imp:.4f})")

print("\n  3. Cross-Validation Results")
for name, res in cv_results.items():
    print(f"     {name}: MAE {res['cv_mae_mean']:.4f} ± {res['cv_mae_std']:.4f}")

print("\n  4. Model Persistence")
for name, path in saved_models.items():
    print(f"     {path.name}")

print("\n  5. VG Node 1: Auxiliary Feature Impact")
print(f"     Weather features improve MAE by: {weather_impact:.4f} ({weather_impact/baseline_mae*100:+.2f}%)")
print(f"     Holiday features improve MAE by: {holiday_impact:.4f} ({holiday_impact/baseline_mae*100:+.2f}%)")

print("\n" + "="*70)
print("NEXT STEPS FOR W7:")
print("  - Hyperparameter tuning for best model")
print("  - SHAP analysis for explainability")
print("  - Residual analysis and error diagnostics")
print("  - Begin Streamlit app development")
print("="*70)